<center><h1> Descriptive Statistics </h1></center>

In [34]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('insurance.csv')
print(df.head(10))

In [ ]:
print('Average Charges by Gender:')
avg_charge_gen = df['charges'].groupby(df['sex']).mean()
print(avg_charge_gen)
print('Difference:')
diff = abs(avg_charge_gen.iloc[0] - avg_charge_gen.iloc[1])
print(diff)

In [ ]:
print('Average BMI by Gender')
avg_bmi_gen = df['bmi'].groupby(df['sex']).mean()
print(avg_bmi_gen)
print('Difference:')
diff = abs(avg_bmi_gen.iloc[0] - avg_bmi_gen.iloc[1])
print(diff)

In [ ]:
print('Average Age by Gender')
avg_age_gen = df['age'].groupby(df['sex']).mean()
print(avg_age_gen)
print('Difference:')
diff = abs(avg_age_gen.iloc[0] - avg_age_gen.iloc[1])
print(diff)

<center><h1> Probability Analysis </h1></center>

In [ ]:
# print((df['smoker'] == 'yes').head()) # this method we use a lot to convert anything bi-categorical to true/false
p_smoking = (df['smoker'] == 'yes').mean()

# create filtered datasets, think we will use them a lot in the future...
# we get true/false dataset using df['sex'] == 'male' and by using it inside df[...] 
# we get dataset but only the records where filter gave true
df_males = df[df['sex'] == 'male']
df_females = df[df['sex'] == 'female']

p_smoking_male = (df_males['smoker'] == 'yes').mean()
p_smoking_female = (df_males['smoker'] == 'yes').mean()

# but to look better we will use formatting    :.2% 
# :   gives python command to enter settings mode
# .2  to change accuracy to 2 digits after point
# %   makes *100 and adds % at the end
print(f'Probability that the client smokes: {p_smoking:.2%}')
print(f'Probability that the male client smokes: {p_smoking_male:.2%}')
print(f'Probability that the female client smokes: {p_smoking_female:.2%}')

<center><h1> Sampling & Confidence Intervals </h1></center>

In [ ]:
sampled_df = df.sample(n = 100, random_state = 42) # 42 for reproducability

# to calculate CI we could use numpy library we have imported at the start. (for np.sqrt())
# std can be calculated using .std()
# std = standard deviation. it shows in how much of a range can number go from mean.
# low std is observed when you have 42 students of the same height
# high std is observed in 5th grade when girls way taller than boys, and generally height is ...
# ... distributed as with some random number generator

sdf_charge_mean = sampled_df['charges'].mean() #sdf = sampled data frame
sdf_charge_std = sampled_df['charges'].std()
print(sdf_charge_mean)
print(sdf_charge_std)
# std that close to mean is REALLY BAD THING. It means that mean gives nothing.
# You can ask 2 clients, and the charge amount they pay can differ VERY...
# ... VERY much....

# Calculating SE - Standard Error
sdf_len = len(sampled_df)
sdf_se = sdf_charge_std / np.sqrt(sdf_len)
print(f"Standard error of the dataframe's sample: {sdf_se:.2f}")
# SE of such size means that datasets size can't provide confident mean
# (wow, who could've imagined)

# To be confident by 95% that mean for the whole data lies in this range we do next:
ci_lower = sdf_charge_mean - (1.96 * sdf_se) # 1.96 to get 5 % +- range of 100% ( from 97.5 to 102.5).
ci_upper = sdf_charge_mean + (1.96 * sdf_se)

print(f"Sample mean: {sdf_charge_mean:.2f}")
print(f"95% Confidence Interval: [{ci_lower:.2f}, {ci_upper:.2f}]")

In [ ]:
# Bootstrapping.
bootstrap_means = []

# 1000 times is a standard for the bootstrapping.
for i in range(1000):
    boot_sample = sampled_df['charges'].sample(n=100, replace=True) # replace is the main feature used in bootstrap
                                                                    # we imitate situation when the same client can be drawn 3 times
                                                                    # and 0 times at the same time....
    bootstrap_means.append(boot_sample.mean())

ci_lower = np.percentile(bootstrap_means, 2.5)
ci_upper = np.percentile(bootstrap_means, 97.5)

print(f"Bootstrap 95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]")

We see pretty good result if CI of bootstrap is well equal to one we got for the sample....

<center><h1> Hypothesis Testing </h1></center>

In [ ]:
# we will do a t-test. For the start let's compare avg charge paid.
# t test makes the next assumption:
# there are significant difference if range between noise and
# groups means' differ much.
# In other words: we test if difference in males and females mean for the...
# ...charge they pay is happened due to the noise in the dataset, or ...
# ... there are real difference that is statistically important
male_charges = df[df['sex'] == 'male']['charges']
female_charges = df[df['sex'] == 'female']['charges']

t_stat, p_val = stats.ttest_ind(male_charges, female_charges)

print(f"t-statistic: {t_stat:.4f}") # value greater than 2 means that difference is significant. 2 means two standard errors.
print(f"p-value: {p_val:.4f}") # p-value means the next: (printed...)
print(f"There are only {p_val:.2%} chance of getting such difference on random...")

print("H0 suggets, that there are no difference in charge that male and female client pays.")
p_val_treshold = 0.05
if p_val < p_val_treshold:
    print("We are rejecting H0 hypothesis, the difference can't be ommited.")
else:
    print("We have failed to reject H0 hypothesis, the difference is caused by the dataset size.")


print('---')

# But this test is not what I wanted to do. Certainly it is important to find that there are gender difference...
# But I really wanted to see if less than 1 year age gap between smokers and non-smokers is real or just 
# random noise...

smokers_age = df[df['smoker'] == 'yes']['age']
nonsmokers_age = df[df['smoker'] == 'no']['age']

t_stat, p_val = stats.ttest_ind(smokers_age, nonsmokers_age)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.4f}")

print("H0 suggets, that there are no difference in avg age of smoking and nonsmoking clients.")
p_val_treshold = 0.05
if p_val < p_val_treshold:
    print("We are rejecting H0 hypothesis, the difference can't be ommited.")
else:
    print("We have failed to reject H0 hypothesis, the difference is caused by the dataset size.")

***So it is the noise after all...***

<center><h1> Effect Size </h1></center>

**Standard Deviation**

Formula of the std:

$s^2 = \frac{\sum (x - \bar{x})^2}{n - 1}$

What we really do here is simple:

1) We take each and every unit of our **sample**, for this example let's consider it to be some person's height. Let it be $x$
2) $\bar{x}$ is mean height that we have calculated. By taking $x - \bar{x}$ we compute difference between everyone's height from the mean height of our **sample**.
3) For the next 2 reasons: First is to amplify any huge difference, and make both $-10$ and $+10$ cm count we take this difference in the power of 2
4) Noted how I explicitly said **sample**? That is beacause we are not sure if our dataset (amount of people) is sufficient to make a point here - We devide the whole equation by $(n-1)$. With this we say that we accept that we may be wrong, so if height does wiggle alot in its value among all those people we have gathered, we amplify it, to be sure that we are certain that it is nearest accurate value of the variance in height that we may calculate.


**Pooled Standard Deviation**

Now lets split our initial dataset of people in $2$ groups: $1$ for females, and other for males.
If we calculate $STD$ for both of the groups we would get $2$ different results. In order to be able to universaly measure them, and get the singe **"noise of the group"** value, we apply the next equation:

$Pooled\_STD = \sqrt{\frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}{n_1 + n_2 - 2}}$

$s_1^2$ and $s_2^2$ are the variances that is they are squared standard deviations. We need them as normally you can't just take 2 stds and sum them. They don't scale linearly, while Variances do infact. Why is that? Because when you take 2 group and their chaos, to measure their total contribution right you have to sum their squares. It comes from the statement that they have independent different directions, that are not opposite, but more like in a way of 2 sides of the triangle. To compute the third one, you'd have to use pithagoras theorem. $c^2 = a^2 + b^2$.

$n-1$ as well as $n_1 + n_2 - 2$ are used to artificially amplify the result, so that we don't underestimate the uselessness of our calculation if our value is compared to the **true mean** value.

Do the $\sqrt$ at the end of the whole equation to get back to stds, not variances.

**Cohen's D**

Дескриптив статистика, это то, что я говорю по факту и удтверждаю.

In [ ]:
male_charges = df[df['sex'] == 'male']['charges']
female_charges = df[df['sex'] == 'female']['charges']

mal_std = male_charges.std()
fem_std = female_charges.std()

mal_var = male_charges.var()
fem_var = female_charges.var()

mal_m = male_charges.mean()
fem_m = female_charges.mean() # means

mal_n = len(male_charges) #compute number of records of the male dataset
fem_n = len(female_charges) #compute number of records of the female dataset

pooled_std = np.sqrt(((mal_n-1)*male_charges.var() + (fem_n-1)*female_charges.var()) / (mal_n + fem_n - 2)) # var() is std()^2

cohens_d = (mal_m - fem_m) / pooled_std
print(cohens_d)

# BUT IT IS CORRECT IF WE CONSIDER THAT OUR DATA SET IS NOT FULL, BUT OUR INSURANCE COMPANY DOESN'T CARE ABOUT THE CLIENTS THAT DON'T EXIST!
real_cohens_d = (mal_m - fem_m) / np.sqrt((mal_var + fem_var) / 2)
print(real_cohens_d)

if real_cohens_d < 0.2:
    print("We don't care, the difference is insufficient.")
elif real_cohens_d < 0.5:
    print("We are a bit worried, the difference is present, tho it is not large yet.")
elif real_cohens_d < 0.8:
    print("We are worried, the difference is present, it can't be ommited. It is important")
else:
    print("The difference is surely practically significant.")
#still no difference. There are NO PRACTICAL SIGNIFICANCE IN THE STATISTICAL DIFFERENCE BETWEEN CHARGE GOT FROM FEMALE AND MALE CLIENTS.

In [ ]:
smoker_charges = df[df['smoker'] == 'yes']['charges']
nonsmoker_charges = df[df['smoker'] == 'no']['charges']

t_stat, p_val = stats.ttest_ind(smoker_charges, nonsmoker_charges)

print(f"t-statistic: {t_stat:.4f}") # value greater than 2 means that difference is significant. 2 means two standard errors.
print(f"p-value: {p_val:.4f}") # p-value means the next: (printed...)
print(f"There are only {p_val:.2%} chance of getting such difference on random...")

print("H0 suggets, that there are no difference in charge that smoking and nonsmoking client pays.")
p_val_treshold = 0.05
if p_val < p_val_treshold:
    print("We are rejecting H0 hypothesis, the difference can't be ommited.")
else:
    print("We have failed to reject H0 hypothesis, the difference is caused by the dataset size.")

real_cohens_d = (smoker_charges.mean() - nonsmoker_charges.mean()) / np.sqrt((smoker_charges.var() + nonsmoker_charges.var()) / 2)
print(cohens_d)

if real_cohens_d < 0.2:
    print("We don't care, the difference is insufficient.")
elif real_cohens_d < 0.5:
    print("We are a bit worried, the difference is present, tho it is not large yet.")
elif real_cohens_d < 0.8:
    print("We are worried, the difference is present, it can't be ommited. It is important")
else:
    print("The difference is surely practically significant.")

<center><h1> Correlation & Visualization </h1></center>

In [ ]:
df = pd.read_csv('insurance.csv')
print(df.head(5))
# So we have 4 numerical columns where we can look for the correlations.

In [ ]:
age = df['age']
bmi = df['bmi']
chl = df['children']
chg = df['charges']

plt.scatter(age, bmi)
plt.title("Age and BMI")
plt.xlabel("Age")
plt.ylabel("BMI")
plt.show()
print(age.corr(bmi))

plt.scatter(age, chl)
plt.title("Age and Children")
plt.xlabel("Age")
plt.ylabel("Children")
plt.show()
print(age.corr(chl))

plt.scatter(age, chg)
plt.title("Age and Charges")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.show()
print(age.corr(chg))

plt.scatter(bmi, chl)
plt.title("BMI and Children")
plt.xlabel("BMI")
plt.ylabel("Children")
plt.show()
print(bmi.corr(chl))

plt.scatter(bmi, chg)
plt.title("BMI and Charges")
plt.xlabel("BMI")
plt.ylabel("Charges")
plt.show()
print(bmi.corr(chg))

plt.scatter(chl, chg)
plt.title("Children and Charges")
plt.xlabel("Children")
plt.ylabel("Charges")
plt.show()
print(chl.corr(chg))

In [ ]:
plt.scatter(age, chg)
plt.title("Age and Charges")
plt.xlabel("Age")
plt.ylabel("Charges")
plt.show()
print(age.corr(chg))

***.corr()*** method from pandas is pretty useful to compute Pearson's correlation coefficent easily. It always yields result between $-1$ and $1$.

$1$ means that values both grow in the same direction (negative or positive) with same power, that is how much grows one, that much the other does.

$0.3$ to $0.5$ means moderate grow trend among 2 numerical features.

$-$ before number means that they do correlate in fact, but as one grows other deplicts.

So as observed, the only good correlation we do have exists between ***age*** and ***charge***. They do grow together, so the older client gets, the more client is charged for the insurance, however the correlation is pretty noise (as we got it is due to the is_smoking factor being absolutely practically significant)

<center><h1> Regression Model </h1></center>

In [ ]:
df = pd.read_csv('insurance.csv')
print(df.head(5))
# So for this we will use the Linear Regression, however...
# ... we will include several columns in our X dataset (that is data we input to the model) ...
# ...and the target will be "charges"

In [ ]:
df_encoded = pd.get_dummies(df, columns=['sex', 'smoker'], drop_first=True).dropna()
df_encoded.head()

In [ ]:
x = df_encoded[['age', 'bmi', 'children', 'sex_male', 'smoker_yes']].dropna()
y = df_encoded['charges'].dropna()

model = LinearRegression()
model.fit(x,y)
model.predict(x)
r2 = model.score(x, y)
print(f"Model have scored: {r2:.2f}! It's pretty decent results! Model can explain 75% of charges, while the rest 25% are caused by the noise. 1.0 score is practically impossible")
print(f"For each +1 year of age, the charge grows by {model.coef_[0]:.2f}")
print(f"For each +1 to BMI, the charge grows by {model.coef_[1]:.2f}")
print(f"For each additional children, the charge grows by {model.coef_[2]:.2f}")
print(f"If client is female, the charge grows by {abs(model.coef_[3]):.2f}")
print(f"Because charge changes by {model.coef_[3]:.2f} if the client is male...")
print(female_charges.mean() - male_charges.mean(), "Interesting... On average, females DON'T pay more....")
print(f"If client smokes...  Dear, the charge grows by {model.coef_[4]:.2f}")

<center><h1> Logistic Regression </h1></center>

In [32]:
x = df_encoded[['age', 'bmi', 'children', 'sex_male', 'charges']].dropna()
y = df_encoded['smoker_yes'].dropna()
model = LogisticRegression( max_iter = 500 ) #as 100 iterations of finding best weights using loss fucntion wasn't enough
model.fit(x,y)
model.predict(x)
accuracy = model.score(x, y)
print(f'{accuracy:.3f}')

print((df['smoker'] == 'yes').mean())
print("1 out of 5 clients smokes. Or there is 1 in 5 chance that randomly choosen client smokes")

coefficients = model.coef_[0]

interpretation = pd.DataFrame({
    'Feature': x.columns,
    'Coefficient (w)': coefficients,
    'Odds Ratio (e^w)': np.exp(coefficients) # Odds Ration allows to interpret non-human log coefficent into human-readable OR
})

print(interpretation)

0.957
0.20478325859491778
1 out of 5 clients smokes. Or there is 1 in 5 chance that randomly choosen client smokes
    Feature  Coefficient (w)  Odds Ratio (e^w)
0       age        -0.099621          0.905180
1       bmi        -0.360247          0.697504
2  children        -0.237518          0.788583
3  sex_male         0.499980          1.648689
4   charges         0.000392          1.000392


This score means that out of $1000$ clients, model succesfully predicted $957$ **is_smoking**. In other words, correctly predicted whether client smokes or not.

If OR = 1 -> chances are unaffected
If OR < 1 -> chances are lowered
If OR > 1 -> chances grow

That is: 
1) older clients have less chance that they are smokers
2) with growing BMI, the chances that client smokes depletes
3) the more children client have - the less chances that they smoke
4) if client is male, chances that he smokes grow rapidly
5) with each dollar paid in charge, chances that client smokes grows by $0.03%$
   

<center><h1> Model Validation </h1></center>

In [33]:
x = df_encoded[['age', 'bmi', 'children', 'sex_male', 'charges']].dropna()
y = df_encoded['smoker_yes'].dropna()

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

model_final = LogisticRegression(max_iter=500)
model_final.fit(x_train, y_train)

train_acc = model_final.score(x_train, y_train)
test_acc = model_final.score(x_test, y_test)

print(f"Accuracy on Train: {train_acc:.3f}")
print(f"Accuracy on Test: {test_acc:.3f}")

from sklearn.metrics import confusion_matrix, classification_report

y_pred = model_final.predict(x_test)
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy on Train: 0.956
Accuracy on Test: 0.959
Confusion Matrix:
[[208   6]
 [  5  49]]

Classification Report:
              precision    recall  f1-score   support

       False       0.98      0.97      0.97       214
        True       0.89      0.91      0.90        54

    accuracy                           0.96       268
   macro avg       0.93      0.94      0.94       268
weighted avg       0.96      0.96      0.96       268



If model says that client smokes - there are $89%$ that model doesn't lie (Precision 0.89 for smokers)

Model found $91%$ of all smokers in the training dataset

f1-score is $0.9$, which is both precision and recall combined

<center><h1> Model Comparison </h1></center>

In [35]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)

lr_preds = model_final.predict(x_test)
rf_preds = rf_model.predict(x_test)

comparison_data = {
    'Metric': ['Accuracy', 'F1-Score'],
    'Logistic Regression': [
        accuracy_score(y_test, lr_preds),
        f1_score(y_test, lr_preds)
    ],
    'Random Forest': [
        accuracy_score(y_test, rf_preds),
        f1_score(y_test, rf_preds)
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df)

     Metric  Logistic Regression  Random Forest
0  Accuracy             0.958955       0.970149
1  F1-Score             0.899083       0.928571


Well Random Forest mogs :D

As we have pretty realistic dataset, RF does better job. While regression did good, it can't be better that forest, as forest introduces non-linearity with its model, while logistic makes every prediction kinda linear. In real life there is not that much of a linearity, everything tends to be chaotic, and thats where forest does better job...